In [1]:
from backend import *

gfw_path = "D:/Stockage/GFW/"
astd_path = r"D:\Stockage\ASTD"
parquet_path = "../../examples/data/"

months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

# Periods for load_periods
periods = {
    # 2018: months,
    2019: months,
    2020: months
}

In [2]:
df_ASTD = load_periods('all_segments_periods2019_2020.parquet', source=astd_path, periods= periods, remove_nan_rows="default", usecols="default", progress=True)
df_ASTD = df_ASTD[df_ASTD['astd_cat'] == 'Fishing vessels']
df_ASTD.drop(columns=['astd_cat'], inplace=True)
df_ASTD["date_time_utc"] = pd.to_datetime(df_ASTD["date_time_utc"])

display(df_ASTD)

df_GFW = load_periods_gfw('gfw_2019_2020.parquet', source=gfw_path, periods = periods)
df_GFW['date'] = pd.to_datetime(df_GFW['date'])
float_cols = ['cell_ll_lon', 'cell_ll_lat', 'hours', 'fishing_hours']
df_GFW[float_cols] = df_GFW[float_cols].astype('float32')

display(df_GFW)

Loaded period ../../examples/data/all_segments_periods2019_2020.parquet, parameters ignored


,shipid,date_time_utc,flagname,iceclass,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude
3,1892,2019-01-01 00:00:04+00:00,Russia,FS Ice Class 1C,< 1000 GT,1.028629,540,29.700306,70.626442
4,3009,2019-01-01 00:00:04+00:00,Norway,FS Ice Class 1C,< 1000 GT,1945.192749,371,25.147583,70.990837
7,2946,2019-01-01 00:00:05+00:00,Iceland,FS Ice Class 1C,< 1000 GT,1.057516,119,-23.244274,66.155754
11,3034,2019-01-01 00:00:08+00:00,Iceland,FS Ice Class 1C,1000 - 4999 GT,3.720555,100,-13.740026,65.135429
17,3107,2019-01-01 00:00:14+00:00,Iceland,FS Ice Class 1C,< 1000 GT,1.718225,170,-23.810671,64.921066
...,...,...,...,...,...,...,...,...,...
85472769,1634,2020-12-31 23:59:51+00:00,Russia,FS Ice Class 1B,1000 - 4999 GT,1627.817993,364,10.828934,77.686615
85472770,2113,2020-12-31 23:59:51+00:00,Norway,FS Ice Class 1C,1000 - 4999 GT,993.163025,369,17.505766,74.173950
85472771,70,2020-12-31 23:59:51+00:00,Iceland,FS Ice Class 1C,< 1000 GT,0.551000,360,-22.718353,64.037857
85472774,1199,2020-12-31 23:59:54+00:00,Iceland,FS Ice Class 1C,< 1000 GT,0.849000,661,-22.425282,63.839603


Loaded data ../../examples/data/gfw_2019_2020.parquet, parameters ignored


,date,cell_ll_lat,cell_ll_lon,mmsi,hours,fishing_hours
0,2019-01-01,-74.599998,-106.500000,440347000,0.8363,0.5983
1,2019-01-01,-74.599998,-106.300003,440347000,0.2288,0.0000
2,2019-01-01,-74.599998,-105.699997,440347000,0.2599,0.2599
3,2019-01-01,-74.599998,-106.400002,440347000,0.3008,0.0000
4,2019-01-01,-74.599998,-106.599998,440347000,1.8158,1.5694
...,...,...,...,...,...,...
110192515,2020-12-31,79.199997,8.800000,273418680,0.8986,0.8986
110192516,2020-12-31,79.300003,8.600000,273352280,0.1138,0.1138
110192517,2020-12-31,79.300003,8.400000,273352280,0.8827,0.7363
110192518,2020-12-31,79.300003,8.500000,273352280,1.1450,0.9033


In [3]:
# Create a month field to avoid build_light_multi_track_data and other processes to hang
df_GFW['month'] = df_GFW['date'].dt.to_period('M')

dt = df_ASTD['date_time_utc'].dt
df_ASTD['month'] = dt.to_period('M')
df_ASTD['date'] = dt.floor('D')

del dt

C:\Users\virtu\AppData\Local\Temp\ipykernel_6528\1031239364.py:5: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_ASTD['month'] = dt.to_period('M')


In [4]:
#df_ASTD['month'] = df_ASTD['month'].astype(str)

tracks = load_tracks("20192020fishing.parquet", df_ASTD, return_logs=True)  #, w_speed=0.2, w_time=0.4, w_dist=0.4, max_time_gap_hours=250, max_distance_km=600, return_logs=True)
tracks_log = tracks[1]
tracks = tracks[0]

tracks['month'] = pd.PeriodIndex(tracks['month'], freq='M')

Loaded tracks ../../examples/data/20192020fishing.parquet, parameters ignored
Loaded tracks logs


In [5]:
#Build tracks positions - with region selection
# df_ASTD['month'] = df_ASTD['month'].astype(str)

# build_tracks = tb.build_light_multi_track_data(track_table=tracks, positions_df=df_ASTD, preprocess_positions=True, track_sampling=[0, -1])
#
# build_tracks.to_parquet(parquet_path + "TRACKSBUILD20192020fishing.parquet")

# build_tracks = pd.read_parquet(parquet_path + "TRACKSBUILD20192020fishing.parquet")

build_tracks = df_ASTD.merge(tracks, left_on=["shipid", "month"], right_on=["segment_id", "month"], how="left")

build_tracks

,shipid,date_time_utc,flagname,iceclass,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,month,date,segment_id,track_id
0,1892,2019-01-01 00:00:04+00:00,Russia,FS Ice Class 1C,< 1000 GT,1.028629,540,29.700306,70.626442,2019-01,2019-01-01 00:00:00+00:00,1892,2
1,3009,2019-01-01 00:00:04+00:00,Norway,FS Ice Class 1C,< 1000 GT,1945.192749,371,25.147583,70.990837,2019-01,2019-01-01 00:00:00+00:00,3009,1
2,2946,2019-01-01 00:00:05+00:00,Iceland,FS Ice Class 1C,< 1000 GT,1.057516,119,-23.244274,66.155754,2019-01,2019-01-01 00:00:00+00:00,2946,3
3,3034,2019-01-01 00:00:08+00:00,Iceland,FS Ice Class 1C,1000 - 4999 GT,3.720555,100,-13.740026,65.135429,2019-01,2019-01-01 00:00:00+00:00,3034,4
4,3107,2019-01-01 00:00:14+00:00,Iceland,FS Ice Class 1C,< 1000 GT,1.718225,170,-23.810671,64.921066,2019-01,2019-01-01 00:00:00+00:00,3107,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
30759691,1634,2020-12-31 23:59:51+00:00,Russia,FS Ice Class 1B,1000 - 4999 GT,1627.817993,364,10.828934,77.686615,2020-12,2020-12-31 00:00:00+00:00,1634,2789
30759692,2113,2020-12-31 23:59:51+00:00,Norway,FS Ice Class 1C,1000 - 4999 GT,993.163025,369,17.505766,74.173950,2020-12,2020-12-31 00:00:00+00:00,2113,2890
30759693,70,2020-12-31 23:59:51+00:00,Iceland,FS Ice Class 1C,< 1000 GT,0.551000,360,-22.718353,64.037857,2020-12,2020-12-31 00:00:00+00:00,70,2357
30759694,1199,2020-12-31 23:59:54+00:00,Iceland,FS Ice Class 1C,< 1000 GT,0.849000,661,-22.425282,63.839603,2020-12,2020-12-31 00:00:00+00:00,1199,1935


In [9]:
# with open(parquet_path + "match_gfw.json", "r") as f:
#     data = json.load(f)
lat_threshold = 59

t_astd = 0.9
t_gfw_high = 0.6
t_gfw_low = 0.8
n_small = 20

data = pd.read_csv(parquet_path + f"gfw_threshmatch2019_2020.{lat_threshold}{[t_astd,t_gfw_low,t_gfw_high,n_small]}.csv")
# data = pd.read_csv(parquet_path + f"gfw_strictmatch2019_2020.{lat_threshold}.csv")
# data= data[data['score']>=1.9]
data['month'] = pd.PeriodIndex(data['month'], freq='M')
data

,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,max_score_shipid
0,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,1.832592
1,276806000,2020-10,641,1332,1355,1577,0.844642,0.983026,1.827668,1.827668,1.827668
2,258535000,2020-10,1219,1301,1370,1513,0.859881,0.949635,1.809516,1.809516,1.809516
3,273451570,2020-09,324,1274,1290,1556,0.818766,0.987597,1.806363,1.806363,1.806363
4,277558000,2020-11,2082,1181,1192,1349,0.875463,0.990772,1.866235,1.866235,1.866235
...,...,...,...,...,...,...,...,...,...,...,...
2425,251153000,2020-06,1557,3,3,3,1.000000,1.000000,2.000000,2.000000,2.000000
2426,257046460,2020-10,478,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000
2427,231189000,2020-05,22009,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000
2428,259616000,2019-08,21852,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000


In [10]:
df = data.copy()


print("Total size (unique shipid):", df.shape[0])
print("Total mmsi :", df['mmsi'].nunique())

df_pct = ( # Pourcentage, of how many mmsi have 12, 11, 10 ... matched months
    df
    .groupby('mmsi')['month']
    .nunique()
    .value_counts(normalize=True)
    .mul(100)
    .rename('percent')
    .reset_index(name='percent')
    .rename(columns={'index': 'n_months'})
)

display(df_pct)

df



Total size (unique shipid): 2430
Total mmsi : 187


,month,percent
0,1,18.716578
1,22,16.042781
2,23,6.951872
3,17,6.417112
4,18,6.417112
5,19,6.417112
6,2,6.417112
7,21,5.347594
8,20,3.743316
9,16,3.208556


,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,max_score_shipid
0,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,1.832592
1,276806000,2020-10,641,1332,1355,1577,0.844642,0.983026,1.827668,1.827668,1.827668
2,258535000,2020-10,1219,1301,1370,1513,0.859881,0.949635,1.809516,1.809516,1.809516
3,273451570,2020-09,324,1274,1290,1556,0.818766,0.987597,1.806363,1.806363,1.806363
4,277558000,2020-11,2082,1181,1192,1349,0.875463,0.990772,1.866235,1.866235,1.866235
...,...,...,...,...,...,...,...,...,...,...,...
2425,251153000,2020-06,1557,3,3,3,1.000000,1.000000,2.000000,2.000000,2.000000
2426,257046460,2020-10,478,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000
2427,231189000,2020-05,22009,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000
2428,259616000,2019-08,21852,2,2,2,1.000000,1.000000,2.000000,2.000000,2.000000


In [11]:
merged = data.merge(build_tracks, left_on=['month', 'shipid'], right_on=['month', 'shipid'], how='inner')
merged

,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,...,flagname,iceclass,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,date,segment_id,track_id
0,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,...,Russia,FS Ice Class 1B,1000 - 4999 GT,11.440143,368,15.420208,68.699417,2020-04-01 00:00:00+00:00,1952,1943
1,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,...,Russia,FS Ice Class 1B,1000 - 4999 GT,10.240563,362,15.420485,68.699432,2020-04-01 00:00:00+00:00,1952,1943
2,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,...,Russia,FS Ice Class 1B,1000 - 4999 GT,4.562982,369,15.420260,68.699394,2020-04-01 00:00:00+00:00,1952,1943
3,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,...,Russia,FS Ice Class 1B,1000 - 4999 GT,5.362920,370,15.420372,68.699387,2020-04-01 00:00:00+00:00,1952,1943
4,273217010,2020-04,1952,1426,1451,1678,0.849821,0.982771,1.832592,1.832592,...,Russia,FS Ice Class 1B,1000 - 4999 GT,4.218481,371,15.420268,68.699417,2020-04-01 00:00:00+00:00,1952,1943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14282806,277558000,2019-05,15527,2,2,2,1.000000,1.000000,2.000000,2.000000,...,Lithuania,FS Ice Class 1B,1000 - 4999 GT,5.854236,535,-21.984692,64.067963,2019-05-31 00:00:00+00:00,15527,1078
14282807,277558000,2019-05,15527,2,2,2,1.000000,1.000000,2.000000,2.000000,...,Lithuania,FS Ice Class 1B,1000 - 4999 GT,7.447917,361,-21.984655,64.068016,2019-05-31 00:00:00+00:00,15527,1078
14282808,277558000,2019-05,15527,2,2,2,1.000000,1.000000,2.000000,2.000000,...,Lithuania,FS Ice Class 1B,1000 - 4999 GT,3.012570,360,-21.984646,64.067947,2019-05-31 00:00:00+00:00,15527,1078
14282809,277558000,2019-05,15527,2,2,2,1.000000,1.000000,2.000000,2.000000,...,Lithuania,FS Ice Class 1B,1000 - 4999 GT,2.366595,537,-21.984634,64.067970,2019-05-31 00:00:00+00:00,15527,1078


In [12]:
display(merged.groupby(['mmsi', 'month'])['track_id'].unique().reset_index())
# merged.groupby(['mmsi', 'month'])['track_id'].nunique().reset_index()
merged.groupby('mmsi')['month'].nunique()

,mmsi,month,track_id
0,231004000,2019-01,[124]
1,231004000,2019-06,[1080]
2,231004000,2019-10,[1214]
3,231004000,2019-11,[1214]
4,231004000,2019-12,[1214]
...,...,...,...
2425,331781000,2019-05,[830]
2426,331781000,2019-06,[830]
2427,331781000,2019-07,[830]
2428,331781000,2019-08,[830]


mmsi
231004000    11
231010000     1
231037000     1
231066000    19
231081000     1
             ..
331478000     7
331479000     3
331549000    12
331746000    15
331781000     6
Name: month, Length: 187, dtype: int64

In [13]:
df = merged.copy()

print("Total mmsi :", df['mmsi'].nunique())
df_pct = ( # Pourcentage, of how many mmsi have 1, 2, 3 ... differents tracks over their shipids
    df
    .groupby('mmsi')['track_id']
    .nunique()
    .value_counts(normalize=True)
    .mul(100)
    .rename('percent')
    .reset_index(name='percent')
    .rename(columns={'track_id': 'n_tracks'})
)

display(df_pct)

mask = df.groupby(['mmsi'])['track_id'].transform('nunique') > 1
mask1 = df.groupby(['track_id'])['mmsi'].transform('nunique') > 1

print("Number of mmsi with multiple track_id : ", df[mask]['mmsi'].nunique())
print("Number of track_id within multiple mmsi : ", df[mask1]['mmsi'].nunique())
print("Number of mmsi with multiple track_id, but track_ids only present in this mmsi : ", df[mask & ~mask1]['mmsi'].nunique())


# Only for 1 to 1 mmsi and tracks

# Remove multiple track_id for one mmsi
mask = df.groupby(['mmsi'])['track_id'].transform('nunique') == 1

# Remove multiple mmsi for one track_id
mask1 = df.groupby(['track_id'])['mmsi'].transform('nunique') == 1

df_unique = df[mask1 & mask]

print("Total mmsi with unique track :", df_unique['mmsi'].nunique())

df_unique[['mmsi', 'track_id']].drop_duplicates()

Total mmsi : 187


,n_tracks,percent
0,1,26.737968
1,6,9.625668
2,5,9.625668
3,7,9.625668
4,3,8.556150
5,4,8.021390
6,2,6.417112
7,8,5.882353
8,10,4.812834
9,9,4.278075


Number of mmsi with multiple track_id :  137
Number of track_id within multiple mmsi :  83
Number of mmsi with multiple track_id, but track_ids only present in this mmsi :  134
Total mmsi with unique track : 31


,mmsi,track_id
1334168,331781000,830
1394005,251026000,2113
1716508,331479000,1110
2530236,257089790,1709
4245145,251151000,1945
4737575,316015750,2704
5761219,257032830,2521
9057083,231191000,1532
11759933,259131000,844
12017750,251044110,617


In [14]:
df = df_unique.copy()

print("Total size (unique shipid):", df['shipid'].nunique())
print("Total mmsi :", df['mmsi'].nunique())

df_pct = ( # Pourcentage, of how many mmsi have 12, 11, 10 ... matched months
    df
    .groupby('mmsi')['month']
    .nunique()
    .value_counts(normalize=True)
    .mul(100)
    .rename('percent')
    .reset_index(name='percent')
    .rename(columns={'index': 'n_months'})
)

display(df_pct)

df

Total size (unique shipid): 74
Total mmsi : 31


,month,percent
0,1,61.290323
1,2,16.129032
2,4,6.451613
3,5,6.451613
4,18,3.225806
5,3,3.225806
6,6,3.225806


,mmsi,month,shipid,match_n_mmsi,astd_n_ship,gfw_n_mmsi,ratio_gfw,ratio_astd,score,max_score_mmsi,...,flagname,iceclass,sizegroup_gt,dist_nextpoint,sec_nextpoint,longitude,latitude,date,segment_id,track_id
1334168,331781000,2019-08,4545,862,944,1067,0.807873,0.913136,1.721008,1.721008,...,Denmark,FS Ice Class 1C,1000 - 4999 GT,1937.910400,1136,-30.198799,65.594025,2019-08-01 00:00:00+00:00,4545,830
1334169,331781000,2019-08,4545,862,944,1067,0.807873,0.913136,1.721008,1.721008,...,Denmark,FS Ice Class 1C,1000 - 4999 GT,4065.008789,5426,-30.239120,65.589149,2019-08-01 00:00:00+00:00,4545,830
1334170,331781000,2019-08,4545,862,944,1067,0.807873,0.913136,1.721008,1.721008,...,Denmark,FS Ice Class 1C,1000 - 4999 GT,2897.224854,1600,-30.316240,65.606796,2019-08-01 00:00:00+00:00,4545,830
1334171,331781000,2019-08,4545,862,944,1067,0.807873,0.913136,1.721008,1.721008,...,Denmark,FS Ice Class 1C,1000 - 4999 GT,4857.608887,2599,-30.379009,65.605560,2019-08-01 00:00:00+00:00,4545,830
1334172,331781000,2019-08,4545,862,944,1067,0.807873,0.913136,1.721008,1.721008,...,Denmark,FS Ice Class 1C,1000 - 4999 GT,2476.816895,1306,-30.484213,65.603226,2019-08-01 00:00:00+00:00,4545,830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14280623,257032830,2020-08,18092,4,4,4,1.000000,1.000000,2.000000,2.000000,...,Norway,FS Ice Class 1C,< 1000 GT,3.855000,362,5.843032,62.323921,2020-08-31 00:00:00+00:00,18092,2521
14280624,257032830,2020-08,18092,4,4,4,1.000000,1.000000,2.000000,2.000000,...,Norway,FS Ice Class 1C,< 1000 GT,7.209000,540,5.842906,62.323891,2020-08-31 00:00:00+00:00,18092,2521
14280625,257032830,2020-08,18092,4,4,4,1.000000,1.000000,2.000000,2.000000,...,Norway,FS Ice Class 1C,< 1000 GT,2.374000,539,5.842930,62.323910,2020-08-31 00:00:00+00:00,18092,2521
14280626,257032830,2020-08,18092,4,4,4,1.000000,1.000000,2.000000,2.000000,...,Norway,FS Ice Class 1C,< 1000 GT,1.982000,362,5.842916,62.323895,2020-08-31 00:00:00+00:00,18092,2521


In [15]:
## Find mmsi with different track_id , overlapping

df = merged.copy()

df = df[['mmsi', 'track_id']].drop_duplicates()
dfre = build_tracks[['shipid', 'month', 'track_id']].drop_duplicates()

ret = df.merge(dfre, on='track_id', how='inner')



ret['num'] = ret.groupby(['mmsi', 'month'])['track_id'].transform('nunique')

ret[['mmsi', 'num', 'month']].drop_duplicates()

,mmsi,num,month
0,273217010,1,2020-03
1,273217010,1,2020-04
2,276806000,1,2020-08
3,276806000,1,2020-09
4,276806000,1,2020-10
...,...,...,...
4261,231037000,1,2019-12
4262,231037000,1,2020-01
4263,231037000,1,2020-02
4264,231037000,1,2020-03


In [17]:
# Get longest

err = df_GFW[df_GFW['mmsi'].isin(merged['mmsi'].unique())]
# err['month'] = err['month'].astype(str)
err = err[['mmsi','month', 'cell_ll_lat', 'cell_ll_lon']].drop_duplicates()
# err['total'] = err.groupby(['mmsi'])['mmsi'].transform('count')
ddd = merged[['mmsi', 'month', "shipid", "track_id", 'match_n_mmsi']].drop_duplicates()
eerrr = ddd.merge(err, on=['mmsi', 'month'])
eerrr.drop_duplicates(subset=['cell_ll_lat', 'cell_ll_lon'], inplace=True)
eerrr.groupby(['mmsi', 'month'])['mmsi'].count()


mmsi       month  
231004000  2019-01      5
           2019-06     40
           2020-05      3
           2020-06     13
231066000  2019-01    237
                     ... 
331781000  2019-05     65
           2019-06    195
           2019-07     52
           2019-08    313
           2019-09     16
Name: mmsi, Length: 1083, dtype: int64

In [ ]:
mmsi = 273313560
# track_ids = [188, 201, 576, 876, 1305, 1320
track_ids =merged[merged['mmsi']==mmsi]['track_id'].unique().tolist()

In [ ]:
merged[merged['mmsi']==mmsi]

In [ ]:
tracks[tracks['track_id'].isin(track_ids)]

In [ ]:
# display(merged[merged['mmsi']==mmsi])
display(merged[merged['mmsi']==mmsi][['mmsi', 'month','track_id', 'shipid', 'match_n_mmsi', 'astd_n_ship', 'gfw_n_mmsi']].drop_duplicates())
data[data['mmsi']==mmsi]

In [ ]:
col_lat_astd = 'latitude'
col_lon_astd = 'longitude'

col_lat_gfw = 'cell_ll_lat'
col_lon_gfw = 'cell_ll_lon'


df1 = build_tracks[(build_tracks['track_id'].isin(track_ids))]#& (build_tracks['month']=='2020-07')]
df2 = df_GFW[(df_GFW['mmsi']==mmsi)]# & (df_GFW['month']=='2020-03')]
df2 = df2.drop_duplicates(subset=[col_lon_gfw, col_lat_gfw, 'month'])

# df2 = df2.sample(frac=0.6)

display(df2)
df1

In [ ]:
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

pio.renderers.default = "browser"
# pio.renderers.default = "notebook"

months = pd.to_datetime(
    pd.concat([df1['month'], df2['month']])
).dt.strftime("%Y-%m").unique()
month_colors = px.colors.qualitative.Plotly  # 12 colors

color_map = {m: month_colors[i % len(month_colors)] for i, m in enumerate(sorted(months))}

# Generate the figure
# fig2 = tb.plot_ship_tracks(
#     df1,
#     track_ids=track_ids,
#     show_points=True,
#     show_start_end=True,
#     map_style="open-street-map"
# )

fig2 = go.Figure()

for (month, track_id), df_month in df1.groupby(['month', 'track_id']):
    df_month = df_month.sort_values('date_time_utc')

    fig2.add_trace(go.Scattermap(
        lon=df_month[col_lon_astd],
        lat=df_month[col_lat_astd],
        mode="lines+markers",
        name=f"TRACK {track_id} {month}",
        line=dict(color=color_map[month], width=2),
        marker=dict(size=6),
        customdata=df_month['date_time_utc'],
        hovertemplate=(
            "Date: %{customdata}<br>"
            "Lat: %{lat}<br>"
            "Lon: %{lon}<extra></extra>"
        )
    ))

# requested_ids_str = set(str(x) for x in track_ids)
# tt_subset = tracks[tracks["track_id"].astype(str).isin(requested_ids_str)].copy()
#
#
# fig2 = tb.plot_individual_track(
#     track_id=track_ids[0],
#     track_table=tt_subset,
#     astd_data=df_ASTD,
#     show_segments=True
# )


In [ ]:

for month, month_df in df2.groupby("month"):
    lons_all = []
    lats_all = []
    hover_text = []
    month = month.strftime("%Y-%m")

    for _, row in month_df.iterrows():
        lon0 = row["cell_ll_lon"]
        lat0 = row["cell_ll_lat"]

        lons_all += [lon0, lon0 + 0.1, lon0 + 0.1, lon0, lon0, None]
        lats_all += [lat0, lat0, lat0 + 0.1, lat0 + 0.1, lat0, None]

        hover_text += [f"Month: {month}<br>Day: {row['date']}"]*6 + [None]

    fig2.add_trace(go.Scattermap(
        lon=lons_all,
        lat=lats_all,
        mode="lines",
        line=dict(color=color_map[month], width=1),
        text=hover_text,
        hovertemplate="%{text}<extra></extra>",
        showlegend=True,
        name=f"GFW {month}"
    ))

# for month, month_df in df2.groupby("month"):
#     month_df = month_df.sort_index()  # ensure order
#
#     fig2.add_trace(go.Scattermapbox(
#         lon=month_df["cell_ll_lon"] + 0.05,
#         lat=month_df["cell_ll_lat"] + 0.05,
#         mode="lines+markers",
#         line=dict(color=color_map[month], width=2),
#         marker=dict(size=6),
#         text=[f"Month: {month}<br>Day: {d}" for d in month_df["date"]],
#         hovertemplate="%{text}<extra></extra>",
#         name=f"{month}"
#     ))



fig2.update_layout(
    map=dict(
        style="open-street-map",
        zoom=3,
        center=dict(lat=df1["latitude"].mean(),
                    lon=df1["longitude"].mean())
    ),
    title=f"{mmsi} for {track_ids}"
)

tb.export_figure(fig2, f"outputs/output_{track_ids}.html")
tb.export_figure(fig2, f"outputs/output_{track_ids}.pdf")

fig2.show()

### Paper ready export

In [ ]:
res = 0.1

# Remove unnecessary months
df2bis = df2[df2['month'].dt.strftime("%Y-%m").isin(df1['month'])]
df1bis = df1[df1['track_id'].isin(track_ids)]


pio.renderers.default = "browser"
# pio.renderers.default = "notebook"

lats = df1bis[col_lat_astd].values
lons = df1bis[col_lon_astd].values

fig = go.Figure()

#GFW plot grids
lons_all = []
lats_all = []

for _, row in df2bis.iterrows():
    lon0 = row[col_lon_gfw]
    lat0 = row[col_lat_gfw]

    lons_all += [lon0, lon0+0.1, lon0+0.1, lon0, lon0, None]
    lats_all += [lat0, lat0, lat0+0.1, lat0+0.1, lat0, None]


fig.add_trace(go.Scattermap(
    lon=lons_all,
    lat=lats_all,
    mode="lines",
    line=dict(color="rgba(230, 120, 0, 0.9)", width=1),
    name=f"GFW Cells : {mmsi}"
))


fig.add_trace(go.Scattermap(
    lon=lons,
    lat=lats,
    mode="lines+markers",
    line=dict(color="#444", width=2),
    name=f"Track : {track_ids}"
))


pad = 1

lat_mean = df2bis[col_lat_gfw].mean()

lon_range = df2bis[col_lon_gfw].max() - df2bis[col_lon_gfw].min()
lat_range = df2bis[col_lat_gfw].max() - df2bis[col_lat_gfw].min()

lon_range_corr = lon_range * np.cos(np.radians(lat_mean))

aspect_ratio = lon_range_corr / lat_range

width = 1080
height = int(width / aspect_ratio)


fig.update_layout(
    autosize=False,
    width=width,
    height=height,
    map=dict(
        style="carto-positron",
        bounds=dict(
            west=df2bis[col_lon_gfw].min() - pad,
            east=df2bis[col_lon_gfw].max() + pad,
            south=df2bis[col_lat_gfw].min() - pad,
            north=df2bis[col_lat_gfw].max() + pad
        ),
        center = dict(
            lon=(df2bis[col_lon_gfw].min() + df2bis[col_lon_gfw].max()) / 2,
            lat=(df2bis[col_lat_gfw].min() + df2bis[col_lat_gfw].max()) / 2
        ),
    ),
    legend=dict(
        orientation="v",
        x=1,
        y=0.1,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.6)"
    ),
    font=dict(size=14),
    margin=dict(l=0, r=0, t=0, b=0),
)

# tb.export_figure(fig, f"GFW2ASTD_{mmsi}.html")
pio.write_image(fig, f"outputs/output_{track_ids}.pdf", format="pdf")

fig.show()

In [ ]:
tracks_log

In [ ]:
tracks_log[(tracks_log['from_shipid']==22009) & (tracks_log['to_shipid']==2643)]

In [ ]:
# df_ASTD[df_ASTD['shipid']==323].iloc[-1]

df_ASTD[(df_ASTD['shipid']==2366) & (df_ASTD['month']=="2020-02")].iloc[-1]

In [ ]:
# df_ASTD[df_ASTD['shipid']==9209].iloc[0]
df_ASTD[(df_ASTD['shipid']==2857) & (df_ASTD['month']=="2020-03")].iloc[0]

In [ ]:
tb.find_track_candidates(2366, '2020-02', df_ASTD, max_time_gap_hours=100)

In [ ]:
to_plot = df_ASTD[df_ASTD['shipid'].isin([1505, 4012, 1031, 4859, 4969])].copy()
to_plot['track_id'] = to_plot['shipid']

fig = tb.plot_ship_tracks(to_plot, [1505, 4012, 1031, 4859, 4969], show_start_end=True)
fig.show()

___

In [ ]:
temp = merged[['mmsi', 'month', 'shipid', 'track_id']].drop_duplicates()

# temp_build= build_tracks.drop_duplicates(subset=['track_id', 'month', "shipid"])[['track_id', 'month', 'shipid', 'iceclass']]
# retemp = temp_build.merge(temp, on=['track_id', 'month', 'shipid'], how='left')

out = []

for _mmsi, group in temp.groupby('mmsi'):
    group = group.sort_values('month').copy()

    group['track_changed'] = group['track_id'].ne(group['track_id'].shift())
    group.loc[group.index[0], 'track_changed'] = False

    group['from_shipid'] = group['shipid']
    group['to_shipid'] = group['shipid'].shift(-1)


    # keep only rows where change happens (excluding first month if you want)
    changed = group[group['track_changed']& group['to_shipid'].notna()].copy()
    changed['to_shipid'] = changed['to_shipid'].astype('Int64')

    changed.drop(columns=['track_id', 'track_changed', 'shipid'], inplace=True)

    out.append(changed)

result = pd.concat(out, ignore_index=True)
result

In [ ]:
merged_bis = result.merge(tracks_log, left_on=['from_shipid', 'to_shipid', 'month'], right_on=['from_shipid', 'to_shipid', 'from_month'],how='left')[['mmsi', 'month', 'from_shipid', 'to_shipid',
                                                                                                                                                   'stage', 'reason', 'dt_hours','distance_km_fd', 'implied_v_kmh']]

merged_bis

In [ ]:
merged_bis.groupby('reason').size()

In [ ]:
merged_bis[merged_bis['reason']=="speed_cap"]

In [ ]:
temp_astd = df_ASTD[df_ASTD['month'].dt.strftime("%Y-%m").isin(["2019-04","2019-05"])]
temp_astd

In [ ]:
tb.find_track_candidates(3568, '2019-04', temp_astd)

In [ ]:
-tracks_log[(tracks_log['from_shipid']==3568)]

In [ ]:
merged[merged['mmsi']==231066000][['mmsi', 'month', 'shipid', 'track_id']].drop_duplicates()

In [ ]:
merged[merged['track_id']==880][['mmsi', 'month', 'shipid', 'track_id']].drop_duplicates()

In [ ]:


# Remove multiple mmsi for one track_id
mask1 = merged.groupby(['track_id'])['mmsi'].transform('nunique') == 1

df = merged[mask1]
df[df['mmsi']==258205000][['mmsi', 'month', 'shipid', 'track_id']].drop_duplicates()

In [ ]:
# How many mmsi with 2 folowing months

mask1 = merged.groupby(['mmsi'])['month'].transform('nunique') >= 2
mask = merged.groupby(['mmsi'])['track_id'].transform('nunique') > 1


df = merged[mask1 & mask][['mmsi', 'month', 'shipid', 'track_id']].drop_duplicates()

df

In [2]:
# Check concurrent month if they have different track_id (why are they on different ?)

df = df.sort_values(['mmsi', 'month'])
df['month'] = pd.to_datetime(df['month'])
# compute difference in months within each mmsi
# previous values per mmsi
df['prev_date'] = df.groupby('mmsi')['month'].shift(1)
df['prev_id2'] = df.groupby('mmsi')['track_id'].shift(1)
df['prev_ship'] = df.groupby('mmsi')['shipid'].shift(1)

# compute month difference
df['month_diff'] = (
    df['month'].dt.to_period('M') - df['prev_date'].dt.to_period('M')
).apply(lambda x: x.n if pd.notnull(x) else None)

# condition: consecutive month AND id2 different
df['match'] = (df['month_diff'] == 1) & (df['track_id'] != df['prev_id2'])
df

NameError: name 'df' is not defined

In [ ]:
df[df["match"]==True]